# Chunking retrieval — human evaluation (dense only)

Compare **dense retrieval** across three chunking indexes (`recursive`, `semantic`, `sentence_window`) **without RAG** 

- Same embedding model and search path as [eval/eval_chunking.py](eval/eval_chunking.py) Tier-2 dense search.
- Cosine **scores** from Qdrant are shown as retrieval metadata (how close the query vector is to each chunk).

**Collection used:** collections `hr_rag_recursive`, `hr_rag_semantic`, `hr_rag_sentence_window` in Qdrant (from `02_build_index.ipynb`).

In [2]:
# ── Environment & path setup (same pattern as 05_eval_chunking) ─────────────
import os, sys
from pathlib import Path
from dotenv import load_dotenv

ROOT = Path.cwd().parent  # notebooks/ -> project root
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

load_dotenv(ROOT / ".env")

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
QDRANT_URL     = os.getenv("QDRANT_URL", "http://localhost:6333")
QDRANT_API_KEY = os.getenv("QDRANT_API_KEY", "")

print("OPENAI_API_KEY set:", bool(OPENAI_API_KEY))
print("QDRANT_URL       :", QDRANT_URL)

OPENAI_API_KEY set: True
QDRANT_URL       : https://38718a4e-130d-4ef7-be90-0c1893444e4c.us-east4-0.gcp.cloud.qdrant.io


In [3]:
# ── Clients (embeddings + Qdrant only — no SentenceTransformer) ────────────
from openai import OpenAI
from qdrant_client import QdrantClient
import pandas as pd
from IPython.display import display
import config

oai    = OpenAI(api_key=OPENAI_API_KEY)
qdrant = QdrantClient(url=QDRANT_URL, api_key=QDRANT_API_KEY or None)

STRATEGIES = ["recursive", "semantic", "sentence_window"]
TOP_K      = 5
SNIP_LEN   = 550  # chars of chunk text to show

from eval.eval_chunking import detect_available_collections

avail = set(detect_available_collections(qdrant))
missing = [s for s in STRATEGIES if s not in avail]
if missing:
    print("WARNING: these collections are missing or empty:", missing)
else:
    print("Collections OK:", sorted(avail.intersection(set(STRATEGIES))))

ACTIVE_STRATEGIES = [s for s in STRATEGIES if s in avail] if avail else STRATEGIES
if not ACTIVE_STRATEGIES:
    ACTIVE_STRATEGIES = STRATEGIES
if len(ACTIVE_STRATEGIES) < len(STRATEGIES):
    print("Using strategies:", ACTIVE_STRATEGIES)

Collections OK: ['recursive', 'semantic', 'sentence_window']


In [4]:
# ── Dense retrieval: one embedding per query, then search each collection ────
# Matches pipeline dense search; avoids triple OpenAI calls per query.
import html

from IPython.display import HTML, display
from pipeline.retriever import get_dense_vec


def display_side_by_side_strategies(cols: dict, strategies: list[str]) -> None:
    """Show each column as pre-wrapped text (pandas HTML tables hide newlines and truncate)."""
    ths = "".join(
        f"<th style='text-align:left;padding:8px;background:#f6f8fa;border:1px solid #ccc'>{html.escape(s)}</th>"
        for s in strategies
    )
    tds = []
    for s in strategies:
        body = html.escape(cols.get(s, "(missing)"))
        tds.append(
            f"<td style='vertical-align:top;border:1px solid #ccc;padding:10px;width:33%'>"
            f"<pre style='white-space:pre-wrap;word-break:break-word;margin:0;"
            f"font-family:system-ui,Segoe UI,sans-serif;font-size:13px;line-height:1.45'>{body}</pre></td>"
        )
    display(
        HTML(
            "<table style='border-collapse:collapse;width:100%;table-layout:fixed'>"
            f"<thead><tr>{ths}</tr></thead><tbody><tr>{''.join(tds)}</tr></tbody></table>"
        )
    )


def dense_topk(query: str, collection_name: str, vec: list | None = None):
    """Return formatted chunk dicts for top_k hits (same as retriever dense path)."""
    if vec is None:
        vec = get_dense_vec(query, oai)
    points = qdrant.query_points(
        collection_name=collection_name,
        query=vec,
        using="dense",
        limit=TOP_K,
        with_payload=True,
    ).points
    rows = []
    for rank, r in enumerate(points, 1):
        p = r.payload
        sc = getattr(r, "score", None)
        rows.append({
            "rank": rank,
            "score": round(float(sc), 4) if isinstance(sc, float) else None,
            "chunk_id": p.get("chunk_id"),
            "source": p.get("source"),
            "title": p.get("title"),
            "category": p.get("category"),
            "text": (p.get("text") or "")[:SNIP_LEN],
        })
    return rows


def format_top1_cell(rows: list) -> str:
    if not rows:
        return "(no results)"
    h = rows[0]
    parts = [
        f"score={h['score']}",
        f"source={h.get('source')}",
        f"category={h.get('category')}",
        f"title={h.get('title')}",
        f"chunk_id={h.get('chunk_id')}",
        "",
        h.get("text", ""),
    ]
    return "\n".join(str(p) for p in parts if p is not None)

In [5]:
# ── Five fixed questions (diverse topics; aligned with eval failure cases) ──
QUERIES = [
    # flexible_work
    "What are the three types of Flexible Work Arrangements under Singapore's TG-FWAR?",
    # mental_health
    "What should employers do to support employee mental well-being at the organisational level?",
    # workplace_fairness / TAFEP
    "What are the five principles of Fair Employment Practices in Singapore?",
    # harassment
    "What counts as workplace harassment and what values should organisations adopt to prevent it?",
    # crisis_support
    "What crisis support services are available in Singapore for employees experiencing mental health issues?",
]

In [6]:
# ── Side-by-side: top-1 per strategy for each of the five questions ────────
for qi, query in enumerate(QUERIES, 1):
    print("=" * 80)
    print(f"Query {qi}/5")
    print(query)
    print("=" * 80)

    vec = get_dense_vec(query, oai)
    cols = {}
    for strat in ACTIVE_STRATEGIES:
        cname = f"{config.COLLECTION_PREFIX}_{strat}"
        rows = dense_topk(query, cname, vec=vec)
        cols[strat] = format_top1_cell(rows)

    display_side_by_side_strategies(cols, ACTIVE_STRATEGIES)
    print()

Query 1/5
What are the three types of Flexible Work Arrangements under Singapore's TG-FWAR?


recursive,semantic,sentence_window
"score=0.7537 source=MOM category=flexible_work title=Tripartite Guidelines on Flexible Work Arrangement Requests (TG-FWAR) chunk_id=c91f2ed6-4677-624b-c37e-128420d255a3 TYPES OF FWAs 4. FWAs are work arrangements where employers and employees agree to a variation from the standard work arrangement. FWAs may fall into one or more of these three broad categories: FLEXI-PLACE FLEXI-TIME FLEXI-LOAD where employees work where employees where employees work flexibly at different timings work flexibly with flexibly from different with no changes to total different workloads and locations aside from their work hours and workload with commensurate usual office location (e.g. flexi-hours, staggered remuneration (e.g. jo","score=0.7311 source=MOM category=flexible_work title=FAQs on TG-FWAR chunk_id=6a61586e-9450-6b19-690f-9acb7d77f6a6 FAQs on Tripartite Guidelines on Flexible Work Arrangement Requests Segment 1: General FAQs on TG-FWAR ........................................................................................... 2 1. If my company already has a flexible work arrangement (FWA) policy, would the employer still have to consider employees’ requests for additional FWAs or variations of the company’s FWA policy? ........................................................................................................................................... 2 2. If an employ","score=0.7364 source=TAFEP category=workplace_fairness title=None chunk_id=931fb727-9e03-e958-feba-e191ae9696e8 Flexible Work Arrangements The Tripartite Standard on Flexible Work Arrangements (TS-FWA) was discontinued on 1 December 2024, replaced by the Tripartite Guidelines on Flexible Work Arrangement Requests (TG-FWAR)."



Query 2/5
What should employers do to support employee mental well-being at the organisational level?


recursive,semantic,sentence_window
"score=0.744 source=WSH_Council category=mental_health title=Handbook on Supporting Employees Mental Health (Oct 2025) chunk_id=1126c807-821d-50f6-d603-be910435aa84 The following recommendations outline good practices that employers can consider voluntarily adopting based on their operational needs, feasibility and impact to organisation. Employers can also tap on external support for employees at risk of, or recovering from mental health conditions, including leveraging established mental health service providers (refer to “Resources” section at page 29 onwards). Recommendation 1: Nurture a Positive Workplace Culture (applicable for all employees) The foundation for a workplace conducive to mental health","score=0.7287 source=WSH_Council category=mental_health title=None chunk_id=a92dfb19-d04a-d0fe-6469-47c842314c7d What I can do as an Employer Learn what you can do to kickstart your organisation journey or advance further as a progressive employer Step 1: Understand the situation • Use iWorkHealth, a free online survey tool to help you find out your workforce’s overall state of mental well-being, and key workplace stressors affecting your employees’ mental well-being.Step 2: Equip your team with knowledge and skills • Organise talks or workshops to raise awareness. The Total WSH Programme offers free mental well-being workshops to help your employees know",score=0.7504 source=WSH_Council category=mental_health title=None chunk_id=f287647c-48a5-c2bd-b8de-70e49c2e3f19 Understand the importance of looking after employee mental well-being and check out what other companies have done Having good workplace mental well-being boosts productivity and can have positive outcomes for both people and organisations.



Query 3/5
What are the five principles of Fair Employment Practices in Singapore?


recursive,semantic,sentence_window
score=0.7917 source=TAFEP category=workplace_fairness title=Tripartite Guidelines on Fair Employment Practices chunk_id=6e023b37-2007-f017-55e0-9b6425515d60 01 T FA R I I R PA E R M TI P T L E O G Y U M ID EN EL T I N P E R S A O CT N I CES FAIR T E R M IP FA A PIL R R T O T E RI YM TIP M EA P L RG EO T N UI Y T TM IED P G E E RN UL AT IIND CP EE RT LS AI IN C C OE ET NS SI C O E N S0 2 Principles of Fair Introduction Employment Practices The five principles of Fair Employment Practices are: Singapore is a meritocratic society and implementing fair and merit-based employment practices is the right thing to do. Singapore also has a diverse a. Recruit and select employees on the basis of merit (such,score=0.7466 source=TAFEP category=workplace_fairness title=Tripartite Guidelines on Fair Employment Practices chunk_id=cdf65a48-73dd-2c13-8ab9-bb080bb5ba37 Tripartite Guidelines On Fair Employment Practices CONTENTS Introduction 01 Principles of Fair Employment Practices 02 Consistent and Fair Selection Criteria 03 Hiring and Developing 04 a Singaporean Core Recruitment 05 – Job Advertisements – Job Applications – Job Interviews – Tests Remuneration 12 Performance Appraisal 13 and Promotion Disciplinary Actions and Dismissals 14 Grievance Handling 15 Workplace Harmony 16 Roles of Employers and Employees 17 Conclusion 18 About TAFEP 19 Produced by: tafep.sg Supported by: RReepprriinntteedd iinn AA,score=0.7441 source=TAFEP category=workplace_fairness title=Tripartite Guidelines on Fair Employment Practices chunk_id=13fdfc49-3eed-0b26-a487-e3cf7e33acf9 CONTENTS Introduction 01 Principles of Fair Employment Practices 02 Consistent and Fair Selection Criteria 03 Hiring and Developing 04 a Singaporean Core Recruitment 05 – Job Advertisements – Job Applications – Job Interviews – Tests Remuneration 12 Performance Appraisal 13 and Promotion Disciplinary Actions and Dismissals 14 Grievance Handling 15 Workplace Harmony 16 Roles of Employers and Employees 17 Conclusion 18 About TAFEP 19 01 T FA R I I R PA E R M TI P T L E O G Y U M ID EN EL T I N P E R S A O CT N I CES FAIR T E R M IP FA A PIL R R



Query 4/5
What counts as workplace harassment and what values should organisations adopt to prevent it?


recursive,semantic,sentence_window
"score=0.7228 source=MOM category=harassment title=Tripartite Advisory on Managing Workplace Harassment chunk_id=6543534d-1f6d-c1d9-1fc3-01b29d8b97a5 Common Myths about Workplace Harassment and Core Values Why It Should be Addressed Myth 1: Workplace harassment is a matter of personal Everyone in the organisation has a role to play in ensuring relationships; and not for employers to intervene. that work is conducive, safe and free from harassment. Individuals are responsible for their own conduct at work, Harassment may directly or indirectly cause anxiety to Harassment and should be respectful towards other persons at the employees at the workplace, affecting the morale and workplace. may d","score=0.6873 source=MOM category=harassment title=Tripartite Advisory on Managing Workplace Harassment chunk_id=9165e4b4-573d-25c5-0a94-16983ee7b487 Individuals are responsible for their own conduct at work, Harassment may directly or indirectly cause anxiety to Harassment and should be respectful towards other persons at the employees at the workplace, affecting the morale and workplace. may directly productivity of the organisation. Such behaviour, if not discouraged, may also affect the reputation of the or indirectly The employer plays an important role in determining organisation. the culture of the organisation. It is useful for employers cause anxiety to establish a set of common val",score=0.7009 source=MOM category=harassment title=Tripartite Advisory on Managing Workplace Harassment chunk_id=43ed1317-ec78-ca52-3d21-aff4899ebbf4 3 Common Myths about Workplace Harassment and Core Values Why It Should be Addressed Myth 1: Workplace harassment is a matter of personal Everyone in the organisation has a role to play in ensuring relationships; and not for employers to intervene.



Query 5/5
What crisis support services are available in Singapore for employees experiencing mental health issues?


recursive,semantic,sentence_window
score=0.7573 source=WSH_Council category=mental_health title=None chunk_id=ba60985c-7b37-5684-8e78-92e853cd93bd 2. Holistic employee psychological well-being programmes 3. Peer Support Training Programme | hpb_health_at_work@hpb.gov.sg | HealthServe Ltd | Conducts basic mental health training and talks for migrant workers in Singapore to deal with distress and adjustment issues. Provides peer support leader training to empower migrant workers to provide initial support to peers facing mental health challenges. Conducts Psychological First Aid (PFA) training for supervisors and managers to enhance their ability to spot signs of mental distress in migrant,score=0.7637 source=WSH_Council category=mental_health title=None chunk_id=c2ace592-85f9-73c3-4d6a-a2aaafe80b43 Holistic employee psychological well-being programmes 3. Peer Support Training Programme | hpb_health_at_work@hpb.gov.sg | HealthServe Ltd | Conducts basic mental health training and talks for migrant workers in Singapore to deal with distress and adjustment issues. Provides peer support leader training to empower migrant workers to provide initial support to peers facing mental health challenges. Conducts Psychological First Aid (PFA) training for supervisors and managers to enhance their ability to spot signs of mental distress in migrant wor,"score=0.7617 source=WSH_Council category=mental_health title=Handbook on Supporting Employees Mental Health (Oct 2025) chunk_id=40d24abc-3c1b-432f-8803-c6ed20748ef9 Contact the following: • Police: 999 and • Samaritans of Singapore’s (SOS) 24-hour Hotline: The designated crisis manager, ideally trained in mental health crisis management (including suicide prevention) (See “Training to Better Support Employees at Risk of or Recovering from Mental Health Conditions” (page 34) for the directory of training providers), should stay close and engage the employee empathetically."


Custom Query

In [7]:
CUSTOM_QUERY = "What can I do to improve work-life balance concerns for staff?"

vec = get_dense_vec(CUSTOM_QUERY, oai)
cols = {}
for strat in ACTIVE_STRATEGIES:
    cname = f"{config.COLLECTION_PREFIX}_{strat}"
    rows = dense_topk(CUSTOM_QUERY, cname, vec=vec)
    cols[strat] = format_top1_cell(rows)

display_side_by_side_strategies(cols, ACTIVE_STRATEGIES)

recursive,semantic,sentence_window
"score=0.6373 source=TAFEP category=workplace_fairness title=None chunk_id=dde9c498-09c8-2061-dc23-912f2cb4e639 - Communicate to employees the different forms of employee support schemes and the process to apply for them (e.g. in company’s staff website, HR policy, circular or memo); and - Adopting technological tools to support work-life strategies, where relevant. *Examples include family day, subsidised health screening, staff recreation areas etc. Other examples of employee support schemes can be found here. - Employer has put in place enhanced leave policies for all employees by: - Providing at least two enhanced leave benefits (e.g. compassionate l","score=0.6655 source=NCSS category=mental_health title=None chunk_id=d7be986b-b3a8-cd5d-2336-fca9f23d4fde Reviewing work regularly and showing appreciation can help prevent burnout and manage fatigue. Reach out to your staff and be open to resolving matters with flexibility such as giving your staff a break to handle their family matters. Such flexibility can be extended if there are no urgent matters at hand, and this could allow your staff to have a good rest before being energised for work again. Promote mental well-being Leaders are crucial in helping to co-create a supportive environment, raise understanding of mental health issues, and reduce","score=0.5956 source=WSH_Council category=mental_health title=None chunk_id=29681c5f-72bb-7dfb-a73c-dd71d1fe91bf Step 3: Review benefits, policies, processes • Check out the list of recommendations to focus on reducing work stressors identified from the iWorkHealth report.• Read the Tripartite Advisory on Mental Well-being at Workplaces to find out how mental well-being can be better supported by initiatives for organisations, teams or individuals."


In [12]:
CUSTOM_QUERY2 = "How do I manage staffs who have children and household duties to handle?"

vec = get_dense_vec(CUSTOM_QUERY2, oai)
cols = {}
for strat in ACTIVE_STRATEGIES:
    cname = f"{config.COLLECTION_PREFIX}_{strat}"
    rows = dense_topk(CUSTOM_QUERY2, cname, vec=vec)
    cols[strat] = format_top1_cell(rows)

display_side_by_side_strategies(cols, ACTIVE_STRATEGIES)

recursive,semantic,sentence_window
"score=0.4892 source=TAFEP category=workplace_fairness title=None chunk_id=dde9c498-09c8-2061-dc23-912f2cb4e639 - Communicate to employees the different forms of employee support schemes and the process to apply for them (e.g. in company’s staff website, HR policy, circular or memo); and - Adopting technological tools to support work-life strategies, where relevant. *Examples include family day, subsidised health screening, staff recreation areas etc. Other examples of employee support schemes can be found here. - Employer has put in place enhanced leave policies for all employees by: - Providing at least two enhanced leave benefits (e.g. compassionate l","score=0.4754 source=NCSS category=mental_health title=None chunk_id=d7be986b-b3a8-cd5d-2336-fca9f23d4fde Reviewing work regularly and showing appreciation can help prevent burnout and manage fatigue. Reach out to your staff and be open to resolving matters with flexibility such as giving your staff a break to handle their family matters. Such flexibility can be extended if there are no urgent matters at hand, and this could allow your staff to have a good rest before being energised for work again. Promote mental well-being Leaders are crucial in helping to co-create a supportive environment, raise understanding of mental health issues, and reduce","score=0.4911 source=TAFEP category=workplace_fairness title=None chunk_id=9c80a69d-7e69-742c-79e8-9f8b3a2a6e09 - Employer to discuss suitable arrangements for employees with caregiving* responsibilities that meet the needs of both employers and employees, such as: - Option to reduce work hours upon request (with a commensurate reduction in pay); or - Other forms of flexible work arrangements including staggered start/end times and telecommuting; or - Additional leave provisions; or - Flexible benefits for caregivers *“Caregiving” refers to looking after children or provides assistance to family members who are ill, disabled, or need help with daily acti"


In [13]:
CUSTOM_QUERY3 = "How do I reduce workplace harrassment or bullying within the organisation?"

vec = get_dense_vec(CUSTOM_QUERY3, oai)
cols = {}
for strat in ACTIVE_STRATEGIES:
    cname = f"{config.COLLECTION_PREFIX}_{strat}"
    rows = dense_topk(CUSTOM_QUERY3, cname, vec=vec)
    cols[strat] = format_top1_cell(rows)

display_side_by_side_strategies(cols, ACTIVE_STRATEGIES)

recursive,semantic,sentence_window
"score=0.6562 source=MOM category=harassment title=Tripartite Advisory on Managing Workplace Harassment chunk_id=15779f25-b43c-39f9-2548-d07995fa46f6 communicated through multiple platforms, such as staff induction programmes, company intranet, HR handbooks, notices or posters, and employee briefings. Measures to prevent harassment by external customers/clients It is also important for the management to discuss and reinforce the messages Employers can consider implementing the following measures to reduce regularly at staff meetings to demonstrate their commitment to it. The policy the potential risk of harassment faced by customer-facing staff. should be reviewed regularly and when an incid","score=0.6194 source=MOM category=harassment title=Tripartite Advisory on Managing Workplace Harassment chunk_id=8acc6438-6b00-a02a-be9f-121250d84ccd h. Non-retaliation – The reporting informant must not be ‘victimised’ by the employer following the making of such a report. i. Accountability – There should be clear documentation of each step 1 For more details on WSH risk management, please refer to the WSH Council’s Approved Code of Practice on WSH Risk Management. of the investigation process and thorough records should be kept. 2 The harassment prevention policy can be standalone or incorporated into a broader company safety and health policy. 6 7 The following table provides suggestions",score=0.6362 source=MOM category=harassment title=Tripartite Advisory on Managing Workplace Harassment chunk_id=9f5301ca-fb72-9d17-9fcf-ccadc6014ddb Provide information and training on workplace harassment • display notification that harassment is unacceptable behaviour on corporate website and plasma/LCD displays in premises It is important to ensure that harassment is taken seriously at all levels of the • Increase lighting in and around the workplace organisation.
